# Evaluate Reverse Polish Notation

# Problem Statement

Evaluate the value of an arithmetic expression written in **Reverse Polish Notation (RPN)**.

The expression contains:

- Integers
- Arithmetic operators:

```text
+
-
*
/
```

Each operator applies to the two numbers immediately before it.

For division, truncate the result toward zero.

### Input

A list of tokens representing a valid Reverse Polish Notation expression.

### Output

Return the integer result of evaluating the expression.

### Examples

```text
Input:
["2", "1", "+", "3", "*"]

Output:
9
```

Explanation:

```text
(2 + 1) * 3 = 9
```

---

```text
Input:
["4", "13", "5", "/", "+"]

Output:
6
```

Explanation:

```text
4 + (13 / 5) = 6
```

---

```text
Input:
["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]

Output:
22
```

# Problem Explanation

In normal arithmetic notation, we usually write:

```text
2 + 1
```

In Reverse Polish Notation, the operator comes **after** its operands:

```text
2 1 +
```

For example:

```text
2 1 + 3 *
```

means:

```text
(2 + 1) * 3
```

The important observation is that when an operator appears, it needs the **two most recently available numbers**.

That is exactly what a Stack provides.

```text
Numbers
   ↓
 PUSH

Operator
   ↓
 POP two values
   ↓
 Perform operation
   ↓
 PUSH result
```

Therefore, the expression can be evaluated from left to right using a Stack.

# Input

```python
tokens = ["2", "1", "+", "3", "*"]
```

# Output

```text
9
```

# Brute Force

A less suitable approach would be to repeatedly find an operator and evaluate the two tokens immediately before it.

For:

```text
2 1 + 3 *
```

we first find:

```text
2 1 +
```

and replace it with:

```text
3
```

giving:

```text
3 3 *
```

Then evaluate:

```text
3 3 *
```

giving:

```text
9
```

This requires repeatedly modifying the token sequence.

The expression itself already tells us the correct evaluation order.

A Stack allows us to process every token once.

# Optimal Approach

Use a Stack.

For every token:

### If the token is a number

Convert it to an integer and push it.

```text
token
  ↓
 PUSH
```

### If the token is an operator

Pop the two most recent numbers.

```text
right = pop()
left = pop()
```

Then:

```text
result = left operator right
```

Push the result back.

```text
result
  ↓
 PUSH
```

After processing every token:

```text
Stack top
   ↓
Answer
```

# Important Detail — Operand Order

For:

```text
-
/
```

the order of operands matters.

Suppose the Stack contains:

```text
2
5
```

with `5` at the top.

When processing:

```text
-
```

the first value popped is:

```text
right = 5
```

The second value popped is:

```text
left = 2
```

Therefore:

```text
left - right
```

is:

```text
2 - 5
```

not:

```text
5 - 2
```

The same rule applies to division.

Always write:

```python
right = stack.pop()
left = stack.pop()
```

before performing the operation.

# Algorithm

```text
Create empty Stack

For each token:

    If token is a number:
        Push integer(token)

    Otherwise:

        right = Pop
        left = Pop

        Calculate:
            left operator right

        Push result

Return Stack top
```

In [1]:
class Solution:

    def evalRPN(self, tokens: list[str]) -> int:

        stack = []

        for token in tokens:

            if token not in "+-*/":

                stack.append(int(token))

            else:

                right = stack.pop()
                left = stack.pop()

                if token == "+":
                    result = left + right

                elif token == "-":
                    result = left - right

                elif token == "*":
                    result = left * right

                else:
                    result = int(left / right)

                stack.append(result)

        return stack[-1]

# Dry Run

Input:

```text
["2", "1", "+", "3", "*"]
```

Start:

```text
Stack = []
```

### `2`

Number → Push

```text
[2]
```

### `1`

Number → Push

```text
[2, 1]
```

### `+`

Pop:

```text
right = 1
left = 2
```

Calculate:

```text
2 + 1 = 3
```

Push:

```text
[3]
```

### `3`

Push:

```text
[3, 3]
```

### `*`

Pop:

```text
right = 3
left = 3
```

Calculate:

```text
3 * 3 = 9
```

Push:

```text
[9]
```

End:

```text
Answer = 9
```

# Dry Run — Division

Input:

```text
["4", "13", "5", "/", "+"]
```

Start:

```text
[]
```

Push `4`:

```text
[4]
```

Push `13`:

```text
[4, 13]
```

Push `5`:

```text
[4, 13, 5]
```

Operator `/`:

```text
right = 5
left = 13
```

Calculate:

```text
13 / 5 = 2
```

Push:

```text
[4, 2]
```

Operator `+`:

```text
right = 2
left = 4
```

Calculate:

```text
4 + 2 = 6
```

Final:

```text
[6]
```

Answer:

```text
6
```

# Edge Cases

### Single Number

```text
Input:
["42"]

Output:
42
```

No operators are required.

---

### Negative Number

```text
Input:
["-3", "2", "*"]

Output:
-6
```

Negative numbers must be treated as numbers, not operators.

---

### Subtraction

```text
Input:
["5", "3", "-"]

Output:
2
```

The order is:

```text
5 - 3
```

---

### Division

```text
Input:
["7", "3", "/"]

Output:
2
```

The result is truncated toward zero.

---

### Negative Division

```text
Input:
["-7", "3", "/"]

Output:
-2
```

The result must be truncated toward zero rather than rounded down.

# Common Mistakes

### Mistake 1 — Reversing Operands

Incorrect:

```python
left = stack.pop()
right = stack.pop()

result = left - right
```

This reverses the operands.

Correct:

```python
right = stack.pop()
left = stack.pop()
```

Then:

```python
result = left - right
```

---

### Mistake 2 — Forgetting to Push the Result

After evaluating an operator:

```text
left operator right
```

the result must be pushed back.

Otherwise later operators cannot use it.

---

### Mistake 3 — Treating Negative Numbers as Operators

```text
"-11"
```

is a number.

The operator is:

```text
"-"
```

The token itself must be interpreted correctly.

# Complexity

Let `N` be the number of tokens.

Every token is processed once.

For each operator:

```text
2 POP operations
1 calculation
1 PUSH operation
```

All are O(1).

Therefore:

```text
Time → O(N)
```

The Stack can contain up to `N` values.

Therefore:

```text
Space → O(N)
```

# Comparison

| Approach | Time | Space |
|---|---:|---:|
| Repeated expression reduction | O(N²) | O(N) |
| Stack | O(N) | O(N) |

The Stack approach matches the structure of Reverse Polish Notation directly.

Each operator consumes the two most recently produced values.

# Pattern Recognition

When an expression is written so that:

```text
Operator comes after operands
```

and each operator uses:

```text
The most recent values
```

think:

```text
STACK
```

The pattern is:

```text
Number
   ↓
 PUSH

Number
   ↓
 PUSH

Operator
   ↓
 POP
 POP
   ↓
Calculate
   ↓
 PUSH result
```

This pattern is useful for:

```text
Expression evaluation
Postfix expressions
Calculator problems
Parsing problems
```

# Takeaway

Reverse Polish Notation removes the need for parentheses and operator precedence by placing each operator after its operands.

The Stack naturally handles this order:

```text
Operands
   ↓
 PUSH
   ↓
Operator
   ↓
 POP two values
   ↓
Calculate
   ↓
 PUSH result
```

The key interview detail is operand order:

```text
right = pop()
left = pop()

result = left operator right
```

The final answer is the only value remaining on the Stack.

```text
Time  → O(N)
Space → O(N)
```